# Evaluación del RAG — Proyecto 2 (Librarian)

Este notebook evalúa la calidad del sistema RAG (Retrieval-Augmented Generation) construido
en el Proyecto 2. El pipeline es:

```
Pregunta → Retriever (ChromaDB) → Top-k chunks → LLM → Respuesta + Citas
```

## Qué evaluamos

1. **Retrieval**: ¿los chunks recuperados son relevantes para la pregunta?
   - Precision@k: ¿qué proporción de los k chunks recuperados son relevantes?
   - Recall@k: ¿se recuperaron todos los productos relevantes?

2. **Fidelidad de citas**: ¿la respuesta cite fuentes que realmente existen en los chunks?
   - Citas correctas: referencias a fuentes recuperadas
   - Citas incorrectas: referencias a fuentes NO recuperadas (alucinaciones)

3. **Calidad cualitativa**: revisión manual lado a lado de pregunta → respuesta → fuentes.

## Por qué ChromaDB

ChromaDB es una base de datos vectorial ligera que corre como librería Python
(sin servidor externo). Ideal para prototipos y producción pequeña. Almacena
embeddings y permite búsqueda por similitud coseno, que es exactamente lo que
necesitamos para encontrar chunks "parecidos" a una pregunta.

## Pre-requisitos

- Parquets del Proyecto 1: `chunks.parquet` y `kb_chunks.parquet`
- ChromaDB indexado (`python index_kb.py`)
- Ollama corriendo en `localhost:11434` (o API key de OpenAI/Anthropic)

In [ ]:
import sys
import json
import os

# Añadir el directorio del proyecto al path
sys.path.insert(0, os.path.abspath('..'))

from responder import answer
from eval_rag import evaluate, save_report, citation_fidelity
from retriever import retrieve

print('Módulos cargados correctamente')

## 1. Verificar el índice

Antes de evaluar, confirmamos que ChromaDB tiene datos indexados.

In [ ]:
import chromadb

persist_dir = os.getenv('CHROMA_PERSIST_DIR', './chroma_db')
client = chromadb.PersistentClient(path=persist_dir)

try:
    collection = client.get_collection('kb_chunks')
    count = collection.count()
    print(f'ChromaDB: {count:,} chunks indexados en kb_chunks')
except Exception as e:
    print(f'ERROR: {e}')
    print('Ejecuta primero: python index_kb.py --chunks <ruta>/chunks.parquet --embeddings <ruta>/kb_chunks.parquet')

## 2. Retrieval manual — explorar resultados

Probamos el retriever con preguntas del dominio y revisamos qué chunks devuelve.

In [ ]:
test_query = '¿Qué productos tienen mejor calificación?'

results = retrieve(test_query, k=3)

print(f'Pregunta: {test_query}')
print(f'Resultados: {len(results)} chunks\n')

for i, r in enumerate(results, 1):
    print(f'--- Chunk {i} (score: {r["score"]:.4f}) ---')
    print(f'Product ID: {r["source"]}')
    print(f'Texto: {r["text"][:200]}...\n')

## 3. Respuesta completa con RAG

Probamos la cadena completa: retriever → LLM → respuesta con citas.

In [ ]:
result = answer(
    '¿Qué productos tienen mejor calificación?',
    k=5,
    model='ollama',
)

print('=' * 60)
print('RESPUESTA:')
print('=' * 60)
print(result['answer'])
print()
print(f'Modelo: {result["model_used"]}')
print(f'Chunks recuperados: {len(result["sources"])}')
print(f'Citas verificadas: {len(result["citations"])}')
print()

print('Fuentes:')
for c in result['citations']:
    print(f'  → {c["product_id"]} (chunk: {c["chunk_id"]})')

## 4. Verificación de citas lado a lado

Para cada fuente citada, verificamos que el chunk recuperado contiene
información relevante a la pregunta.

In [ ]:
print('VERIFICACIÓN DE CITAS\n')

for citation in result['citations']:
    chunk_id = citation['chunk_id']
    # Buscar el chunk en los resultados del retriever
    matching = [s for s in result['sources'] if s['chunk_id'] == chunk_id]
    if matching:
        chunk = matching[0]
        print(f'FUENTE: {citation["product_id"]}')
        print(f'Score: {chunk["score"]:.4f}')
        print(f'Texto del chunk:')
        print(f'  {chunk["text"][:300]}...')
        print()

## 5. Evaluación automática (15 preguntas)

Corremos el evaluador completo con 15 preguntas del dominio y generamos
métricas de retrieval y fidelidad de citas.

In [ ]:
# Preguntas del dominio para evaluación
eval_questions = [
    {'question': '¿Qué productos tienen mejor calificación?', 'relevant_products': []},
    {'question': '¿Cuáles son los productos con más reseñas negativas?', 'relevant_products': []},
    {'question': '¿Qué productos tienen problemas de calidad?', 'relevant_products': []},
    {'question': '¿Cuáles son los productos mejor valorados por los clientes?', 'relevant_products': []},
    {'question': '¿Qué productos tienen reseñas mixtas?', 'relevant_products': []},
    {'question': '¿Cuáles son los productos con urgencia crítica?', 'relevant_products': []},
    {'question': '¿Qué productos recomiendan los clientes?', 'relevant_products': []},
    {'question': '¿Cuáles son los productos más populares?', 'relevant_products': []},
    {'question': '¿Qué productos tienen problemas de envío?', 'relevant_products': []},
    {'question': '¿Cuáles son los productos con mejor relación precio-calidad?', 'relevant_products': []},
    {'question': '¿Qué productos tienen quejas sobre durabilidad?', 'relevant_products': []},
    {'question': '¿Cuáles son los productos favoritos en alimentos?', 'relevant_products': []},
    {'question': '¿Qué productos tienen devoluciones frecuentes?', 'relevant_products': []},
    {'question': '¿Cuáles son los productos con mejor empaque?', 'relevant_products': []},
    {'question': '¿Qué productos tienen problemas de tamaño?', 'relevant_products': []},
]

print(f'Evaluando {len(eval_questions)} preguntas...')
report = evaluate(test_cases=eval_questions, k=5, model='ollama')

In [ ]:
print('\nRESUMEN DE EVALUACIÓN\n')
print(json.dumps(report['summary'], indent=2, ensure_ascii=False))

In [ ]:
# Guardar reportes
paths = save_report(report, output_dir='./eval_reports')
print(f'JSON: {paths["json"]}')
print(f'CSV:  {paths["csv"]}')

## 6. Tabla de resultados detallada

Revisamos cada pregunta individualmente con sus métricas.

In [ ]:
import pandas as pd

df_eval = pd.DataFrame(report['results'])

# Seleccionar columnas clave
cols = ['question', 'retrieved_chunks', 'precision_at_k', 'recall_at_k',
        'citations_correct', 'citations_incorrect', 'fidelity_score']
display(df_eval[cols])

## 7. Análisis de fidelidad de citas

Visualizamos la distribución de fidelidad de citas across las preguntas.

In [ ]:
fidelity_scores = [r['fidelity_score'] for r in report['results']]

print(f'Fidelidad promedio: {sum(fidelity_scores)/len(fidelity_scores):.4f}')
print(f'Mínima: {min(fidelity_scores):.4f}')
print(f'Máxima: {max(fidelity_scores):.4f}')
print()

# Preguntas con baja fidelidad
low_fidelity = [r for r in report['results'] if r['fidelity_score'] < 0.8]
if low_fidelity:
    print('PREGUNTAS CON BAJA FIDELIDAD:')
    for r in low_fidelity:
        print(f'  [{r["fidelity_score"]:.2f}] {r["question"]}')
        if r['citations_incorrect'] > 0:
            print(f'         ⚠ {r["citations_incorrect"]} citas incorrectas')
else:
    print('Todas las preguntas tienen fidelidad >= 0.8')

## 8. Comparación de modelos (opcional)

Si tienes API keys configuradas, puedes comparar calidad entre Ollama, OpenAI y Anthropic.

In [ ]:
# Descomentar para comparar modelos (requiere API keys en .env)
#
# models_to_test = ['ollama']
# if os.getenv('OPENAI_API_KEY'):
#     models_to_test.append('openai')
# if os.getenv('ANTHROPIC_API_KEY'):
#     models_to_test.append('anthropic')
#
# comparison = []
# for model in models_to_test:
#     print(f'\nEvaluando modelo: {model}')
#     report_m = evaluate(test_cases=eval_questions, k=5, model=model)
#     comparison.append({
#         'model': model,
#         **report_m['summary']
#     })
#
# df_compare = pd.DataFrame(comparison)
# display(df_compare)

## Conclusiones

Las métricas clave del RAG son:

| Métrica | Qué mide | Objetivo |
|---------|----------|----------|
| Precision@k | Calidad del retrieval | ≥ 0.7 |
| Recall@k | Cobertura del retrieval | ≥ 0.6 |
| Fidelidad de citas | No alucinar fuentes | ≥ 0.9 |

Si la fidelidad es baja, el prompt del LLM necesita ajuste para citar correctamente.
Si la precisión es baja, los embeddings o el modelo de búsqueda necesitan tuning.